# Clase 24: Series de tiempo — datos temporales

## ¿Cómo detectar tendencias y patrones en datos que cambian con el tiempo?

Una serie de tiempo es como leer el diario de una empresa: cada entrada tiene una fecha y un valor. Leer esas entradas en orden cronológico nos permite ver si las ventas crecen, si hay épocas del año con más actividad y qué podemos esperar el próximo mes.

In [ ]:
# Importar librerías
# pandas para manipular datos temporales, matplotlib para visualizar
# statsmodels para la descomposición estacional
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 11
print('Librerías cargadas ✅')

## 1. Cargar el dataset y trabajar con fechas

El primer problema con datos de tiempo es que las fechas suelen llegar como texto. Pandas no puede hacer cálculos con texto, así que lo primero es convertirlas.

In [ ]:
# Cargar dataset de ventas con columna de fecha
# Si el dataset no existe, generamos datos simulados realistas
import os

if os.path.exists('datasets/ventas_tienda.csv'):
    df = pd.read_csv('datasets/ventas_tienda.csv')
    print('Dataset real cargado.')
    print(f'Columnas: {list(df.columns)}')
    print(f'Tipos: {df.dtypes.to_dict()}')
else:
    # Generar 2 años de datos diarios con tendencia y estacionalidad
    np.random.seed(42)
    fechas = pd.date_range(start='2022-01-01', end='2023-12-31', freq='D')
    n = len(fechas)
    
    tendencia = np.linspace(1000, 1400, n)  # crecimiento gradual
    estacionalidad = 200 * np.sin(2 * np.pi * np.arange(n) / 365 - np.pi/2)  # pico en verano
    ruido = np.random.normal(0, 80, n)
    
    ventas = tendencia + estacionalidad + ruido
    ventas = np.clip(ventas, 200, None).round(0)
    
    df = pd.DataFrame({'fecha': fechas.strftime('%Y-%m-%d'), 'ventas': ventas})
    print('Datos simulados: 2 años de ventas diarias con tendencia y estacionalidad.')

df.head()

In [ ]:
# Convertir la columna de fecha de texto a datetime
# Este paso es OBLIGATORIO para todas las operaciones temporales
col_fecha = 'fecha'   # ajusta al nombre real de tu columna de fecha
col_ventas = 'ventas' # ajusta al nombre real de tu columna de ventas

df[col_fecha] = pd.to_datetime(df[col_fecha])
print(f'Tipo antes: object  →  Tipo después: {df[col_fecha].dtype}')

# Establecer como índice y ordenar
df = df.set_index(col_fecha)
df = df.sort_index()

print(f'\nRango de fechas: {df.index.min().date()} → {df.index.max().date()}')
print(f'Total de registros: {len(df)}')
print(f'\nPrimeros registros:')
df.head()

## 2. Explorar la distribución temporal

¿Cuántos registros hay por mes? ¿El dataset cubre todos los meses sin huecos?

In [ ]:
# Contar registros por mes y ver si hay meses sin datos
# resample('M').count() agrupa por mes y cuenta las filas
conteo_mensual = df[col_ventas].resample('M').count()
print('Registros por mes:')
print(conteo_mensual.to_string())

# Graficar para ver si hay meses con muchos o pocos datos
fig, ax = plt.subplots(figsize=(12, 3))
conteo_mensual.plot(kind='bar', ax=ax, color='steelblue', edgecolor='k')
ax.set_xlabel('Mes')
ax.set_ylabel('Número de registros')
ax.set_title('Registros disponibles por mes')
ax.axhline(conteo_mensual.mean(), color='red', linestyle='--', label=f'Promedio ({conteo_mensual.mean():.0f})')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Resample: ventas por mes

Los datos diarios tienen mucho ruido. Agrupar por mes nos da una visión más clara de la evolución.

In [ ]:
# Agregar ventas por distintos periodos
# resample es como groupby pero para tiempo
ventas_diarias   = df[col_ventas]
ventas_semanales = df[col_ventas].resample('W').sum()
ventas_mensuales = df[col_ventas].resample('M').sum()

print('Ventas mensuales:')
print(ventas_mensuales.round(0).to_string())
print(f'\nMes con más ventas:  {ventas_mensuales.idxmax().strftime("%B %Y")}')
print(f'Mes con menos ventas: {ventas_mensuales.idxmin().strftime("%B %Y")}')

# Gráfico de barras mensual
fig, ax = plt.subplots(figsize=(12, 4))
ventas_mensuales.plot(kind='bar', ax=ax, color='coral', edgecolor='k', alpha=0.8)
ax.axhline(ventas_mensuales.mean(), color='navy', linestyle='--',
           label=f'Promedio mensual: {ventas_mensuales.mean():,.0f}')
ax.set_xlabel('Mes')
ax.set_ylabel('Ventas totales')
ax.set_title('Ventas totales por mes')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Promedio móvil: ver la tendencia sin el ruido

Los datos diarios suben y bajan mucho. El promedio móvil "suaviza" esas variaciones para que podamos ver el patrón general.

In [ ]:
# Calcular promedios móviles de distintas ventanas
# rolling(7): promedio de los últimos 7 días
# rolling(30): promedio de los últimos 30 días (más suavizado)
df['media_7d']  = df[col_ventas].rolling(window=7).mean()
df['media_30d'] = df[col_ventas].rolling(window=30).mean()

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(df.index, df[col_ventas],
        label='Ventas diarias', color='steelblue', alpha=0.35, linewidth=0.8)
ax.plot(df.index, df['media_7d'],
        label='Media móvil 7 días', color='red', linewidth=1.8)
ax.plot(df.index, df['media_30d'],
        label='Media móvil 30 días', color='darkgreen', linewidth=2.5)

ax.set_xlabel('Fecha')
ax.set_ylabel('Ventas')
ax.set_title('Ventas en el tiempo — con promedios móviles')
ax.legend()
ax.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('La línea verde (30 días) revela la tendencia de fondo sin ruido diario.')

## 5. Análisis de estacionalidad por mes del año

¿Los diciembres son siempre los mejores meses? ¿Los eneros siempre los peores? Esto se llama estacionalidad.

In [ ]:
# Calcular el promedio de ventas por mes del año (independiente del año)
# Esto revela patrones estacionales que se repiten todos los años
df['mes_num'] = df.index.month
df['anio']    = df.index.year

promedio_por_mes = df.groupby('mes_num')[col_ventas].mean()
nombres_meses = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun',
                 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
promedio_por_mes.index = nombres_meses[:len(promedio_por_mes)]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Ventas mensuales totales
ventas_mensuales.plot(ax=axes[0], marker='o', color='steelblue')
axes[0].set_title('Ventas totales por mes')
axes[0].set_xlabel('Fecha')
axes[0].set_ylabel('Ventas')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(alpha=0.3)

# Promedio por mes del año
colores_meses = ['#2196F3' if v >= promedio_por_mes.mean()
                 else '#FF7043' for v in promedio_por_mes]
axes[1].bar(promedio_por_mes.index, promedio_por_mes.values,
            color=colores_meses, edgecolor='k')
axes[1].axhline(promedio_por_mes.mean(), color='black',
               linestyle='--', label='Promedio anual')
axes[1].set_title('Perfil estacional (promedio por mes)')
axes[1].set_xlabel('Mes')
axes[1].set_ylabel('Promedio de ventas')
axes[1].legend()

plt.tight_layout()
plt.show()

mejor_mes = promedio_por_mes.idxmax()
peor_mes  = promedio_por_mes.idxmin()
print(f'Mes con más ventas en promedio:  {mejor_mes}')
print(f'Mes con menos ventas en promedio: {peor_mes}')

## 6. Descomposición estacional con statsmodels

Este algoritmo separa automáticamente la serie en tendencia + estacionalidad + ruido.

In [ ]:
# Descomposición estacional
# Necesitamos la serie mensual agregada (menos ruido que la diaria)
from statsmodels.tsa.seasonal import seasonal_decompose

serie = ventas_mensuales.dropna()

# period=12 si los datos son mensuales y la estacionalidad es anual
# Si hay menos de 24 meses, usamos period=6 como fallback
periodo = 12 if len(serie) >= 24 else max(2, len(serie) // 2)

resultado = seasonal_decompose(serie, model='additive', period=periodo)

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

resultado.observed.plot(ax=axes[0], title='Serie original', color='steelblue')
resultado.trend.plot(ax=axes[1], title='Tendencia', color='darkgreen')
resultado.seasonal.plot(ax=axes[2], title='Componente estacional', color='orange')
resultado.resid.plot(ax=axes[3], title='Residuo (ruido)', color='gray')

for ax in axes:
    ax.grid(alpha=0.3)

plt.suptitle('Descomposición estacional — Ventas mensuales',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('Interpretación:')
print('  Tendencia:   dirección general de largo plazo')
print('  Estacional:  patrón repetitivo (cada año)')
print('  Residuo:     lo que no explica el modelo (eventos únicos, errores)')

## 7. Pronóstico simple para el próximo mes

In [ ]:
# Comparar tres métodos de pronóstico simple
# Ninguno es perfecto, pero nos dan un punto de partida
n_meses = len(serie)

p_naive    = serie.iloc[-1]
p_promedio = serie.mean()
p_ma3      = serie.tail(3).mean()

print('Pronósticos para el próximo mes:')
print(f'  Naive (último valor observado):  {p_naive:>10,.0f}')
print(f'  Promedio histórico:              {p_promedio:>10,.0f}')
print(f'  Media móvil 3 meses:             {p_ma3:>10,.0f}')

if n_meses >= 13:
    p_seasonal = serie.iloc[-12]
    print(f'  Seasonal naive (hace 12 meses):  {p_seasonal:>10,.0f}')

# Visualizar los últimos 12 meses + pronósticos
ultimos = serie.tail(12)
siguiente_fecha = serie.index[-1] + pd.DateOffset(months=1)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(ultimos.index, ultimos.values, marker='o', color='steelblue',
        label='Ventas observadas', linewidth=2)

for nombre, valor, color in [
    ('Naive', p_naive, 'red'),
    ('Promedio hist.', p_promedio, 'orange'),
    ('MA3', p_ma3, 'green')
]:
    ax.scatter(siguiente_fecha, valor, s=150, color=color,
               zorder=5, label=f'{nombre}: {valor:,.0f}')
    ax.plot([ultimos.index[-1], siguiente_fecha], [ultimos.values[-1], valor],
            '--', color=color, alpha=0.6)

ax.set_title('Últimos 12 meses + Pronósticos')
ax.set_xlabel('Mes')
ax.set_ylabel('Ventas')
ax.legend()
ax.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('\n¿Cuál método usarías? Depende de la tendencia y estacionalidad observadas.')

## 8. Análisis del dataset de retención de clientes

In [ ]:
# Cargar y analizar el dataset de retención de clientes
# Este dataset tiene una columna de tiempo llamada 'mes'
if os.path.exists('datasets/retencion_clientes.csv'):
    df_ret = pd.read_csv('datasets/retencion_clientes.csv')
    print('Dataset real cargado.')
else:
    # Datos simulados: 24 meses de tasa de retención
    np.random.seed(7)
    meses = pd.date_range(start='2022-01-01', periods=24, freq='MS')
    tasa  = np.clip(
        75 + np.linspace(0, 5, 24) +
        3 * np.sin(2 * np.pi * np.arange(24) / 12) +
        np.random.randn(24) * 2, 60, 95
    ).round(1)
    df_ret = pd.DataFrame({'mes': meses.strftime('%Y-%m'), 'tasa_retencion': tasa})
    print('Datos simulados de retención generados.')

# Procesar fechas y graficar
col_mes = 'mes'
df_ret[col_mes] = pd.to_datetime(df_ret[col_mes])
df_ret = df_ret.set_index(col_mes).sort_index()

col_ret = df_ret.select_dtypes(include='number').columns[0]
df_ret['media_3m'] = df_ret[col_ret].rolling(3).mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_ret.index, df_ret[col_ret],
        marker='o', label='Retención mensual', color='purple', alpha=0.7)
ax.plot(df_ret.index, df_ret['media_3m'],
        label='Media móvil 3 meses', color='darkorange', linewidth=2.5)
ax.set_xlabel('Mes')
ax.set_ylabel('Tasa de retención (%)')
ax.set_title('Retención de clientes en el tiempo')
ax.legend()
ax.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f'Tasa máxima: {df_ret[col_ret].max():.1f}%  |  ')
print(f'Tasa mínima: {df_ret[col_ret].min():.1f}%  |  ')
print(f'Tendencia:   {"creciente" if df_ret[col_ret].iloc[-1] > df_ret[col_ret].iloc[0] else "decreciente"}')